# 05d — TerSeg (Dual-Branch ResNet-34 + Swin-Tiny)

**Paper**: Fan et al. — *TerSeg: A dual-branch semantic segmentation network for Mars terrain and autonomous path planning*  
DOI: `10.1016/j.eswa.2025.126397` — Expert Systems With Applications 270 (2025)  

**Arquitectura**:  
- **Branch_C**: ResNet-34 pretrained — extrae features locales  
- **Branch_T**: Swin-Tiny (timm, pretrained) — extrae features globales  
- **FL/FG modules**: fusión ponderada entre ramas (local-first en capas tempranas, global-first en capas tardías)  
- **FLGA decoder**: agrega features multi-escala de distintas granularidades  

**Limitación importante**: Swin-Tiny pretrained requiere input 224×224.  
Solución: interpolación de los feature maps de entrada antes de Branch_T, no de la imagen completa.

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/ai4mars_DL-v3')
    sys.path.append(str(PROJECT_ROOT / 'notebooks'))
else:
    PROJECT_ROOT = Path('__file__').resolve().parent.parent
    if not (PROJECT_ROOT / 'processed').exists():
        PROJECT_ROOT = Path.cwd().parent

print(f'ROOT: {PROJECT_ROOT} | existe: {PROJECT_ROOT.exists()}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights
import timm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from mars_utils import (
    set_seed, load_norm_stats, load_split,
    build_dataloaders, FocalLoss,
    train_one_epoch, evaluate, train_model, run_multi_seed,
    append_benchmark_results, plot_best_seed_curves,
    print_summary_table, visualize_predictions, count_parameters,
    NUM_CLASSES, IGNORE_INDEX, SEEDS, BENCHMARK_CSV, CHECKPOINT_DIR
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuración

In [ ]:
MODEL_NAME   = 'TerSeg'
FAST_SUBSET  = True   # ← True para prueba rápida (~2 min/seed)

LR           = 1e-4
MAX_EPOCHS   = 80
PATIENCE     = 10
BATCH_SIZE   = 16
FOCAL_ALPHA  = 0.25
FOCAL_GAMMA  = 2.0

# Dimensiones del Swin-Tiny pretrained
SWIN_IMG_SIZE = 224   # ← limitación del pretrained; las imágenes se interpolan internamente

print(f'Modo: {"FAST SUBSET" if FAST_SUBSET else "PRODUCCIÓN"}')
print(f'SWIN input interpolado a: {SWIN_IMG_SIZE}×{SWIN_IMG_SIZE}')

## 2. Datos

In [ ]:
df_train, df_val, df_gold = load_split()
mean, std = load_norm_stats()

train_ids = set(df_train['stem'].tolist())
gold_ids  = set(df_gold['stem'].tolist())
assert len(train_ids & gold_ids) == 0, '⚠️ DATA LEAKAGE detectado'
print(f'✅ Train: {len(df_train)} | Val: {len(df_val)} | Gold: {len(df_gold)}')

## 3. Arquitectura — TerSeg

### 3.1 Módulos FL y FG (fusión de ramas)

In [ ]:
class FLModule(nn.Module):
    """
    Focus Local (FL) — fusión donde CNN domina el peso.
    Usado en las capas tempranas (stages 1-2) donde la CNN extrae
    mejor los detalles locales (bordes, texturas, colores).
    
    f' = r' ⊕ (Conv(Concat(t', r')) ⊗ r')
    """
    def __init__(self, cnn_ch: int, swin_ch: int, out_ch: int):
        super().__init__()
        self.align = nn.Conv2d(swin_ch, cnn_ch, 1)  # alinear canales Swin → CNN
        self.fuse  = nn.Sequential(
            nn.Conv2d(cnn_ch * 2, cnn_ch, 1, bias=False),
            nn.Conv2d(cnn_ch, cnn_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(cnn_ch), nn.ReLU(inplace=True)
        )
        self.proj  = nn.Conv2d(cnn_ch, out_ch, 1) if cnn_ch != out_ch else nn.Identity()

    def forward(self, r, t):
        # r: feature CNN [B, cnn_ch, H, W]
        # t: feature Swin [B, swin_ch, H', W'] — puede tener distinto tamaño
        t_aligned = self.align(
            F.interpolate(t, size=r.shape[-2:], mode='bilinear', align_corners=False)
        )
        gate = torch.sigmoid(self.fuse(torch.cat([r, t_aligned], dim=1)))
        out  = r + gate * r
        return self.proj(out)


class FGModule(nn.Module):
    """
    Focus Global (FG) — fusión donde Swin Transformer domina el peso.
    Usado en capas tardías (stages 3-4) donde el Transformer captura
    mejor el contexto global y las dependencias de largo alcance.
    
    f'' = t'' ⊕ (Conv(Concat(r'', t'')) ⊗ t'')
    """
    def __init__(self, cnn_ch: int, swin_ch: int, out_ch: int):
        super().__init__()
        self.align = nn.Conv2d(cnn_ch, swin_ch, 1)  # alinear canales CNN → Swin
        self.fuse  = nn.Sequential(
            nn.Conv2d(swin_ch * 2, swin_ch, 1, bias=False),
            nn.Conv2d(swin_ch, swin_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(swin_ch), nn.ReLU(inplace=True)
        )
        self.proj  = nn.Conv2d(swin_ch, out_ch, 1) if swin_ch != out_ch else nn.Identity()

    def forward(self, r, t):
        # r: feature CNN  [B, cnn_ch, H, W]
        # t: feature Swin [B, swin_ch, H', W']
        r_aligned = self.align(
            F.interpolate(r, size=t.shape[-2:], mode='bilinear', align_corners=False)
        )
        gate = torch.sigmoid(self.fuse(torch.cat([r_aligned, t], dim=1)))
        out  = t + gate * t
        return self.proj(out)

In [ ]:
class FLGADecoder(nn.Module):
    """
    Focus on Local and Global feature Aggregation (FLGA).
    Agrega features de múltiples granularidades en el decoder:
    1. Upsample todos los feature maps a H/2×W/2
    2. Blend con Conv3×3
    3. Fusión espacial (concatenación + reducción 1×1)
    4. Fusión posicional (suma elemento a elemento)
    5. Interacción espacial-posicional (multiplicación + suma)
    6. Mejora de features finas (concatenar con conv1 del backbone + upsample final)
    """
    def __init__(self, in_channels_list: list, c1_ch: int,
                 mid_ch: int, num_classes: int):
        super().__init__()
        n = len(in_channels_list)

        # Blend: ajustar canales de cada nivel a mid_ch
        self.blends = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(ch, mid_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True)
            ) for ch in in_channels_list
        ])

        # Fusión espacial: n ramas concatenadas → mid_ch
        self.spatial_fuse = nn.Sequential(
            nn.Conv2d(mid_ch * n, mid_ch, 1, bias=False),
            nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True)
        )

        # Mejora fina: concatenar con c1 (capa 1 del backbone)
        self.c1_reduce = nn.Sequential(
            nn.Conv2d(c1_ch, mid_ch // 4, 1, bias=False),
            nn.BatchNorm2d(mid_ch // 4), nn.ReLU(inplace=True)
        )
        self.final_conv = nn.Sequential(
            nn.Conv2d(mid_ch + mid_ch // 4, mid_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True)
        )
        self.head = nn.Conv2d(mid_ch, num_classes, 1)

    def forward(self, fused_list: list, c1: torch.Tensor, out_size: tuple):
        # out_size: (H, W) de la imagen original
        target_h = out_size[0] // 2
        target_w = out_size[1] // 2

        # 1. Upsample + blend
        blended = []
        for feat, blend in zip(fused_list, self.blends):
            up = F.interpolate(feat, size=(target_h, target_w),
                               mode='bilinear', align_corners=False)
            blended.append(blend(up))

        # 2-3. Fusión espacial
        FC = self.spatial_fuse(torch.cat(blended, dim=1))

        # 4. Fusión posicional
        FA = sum(blended)

        # 5. Interacción
        FI = FC + FC * FA

        # 6. Mejora con features finas (c1 del backbone)
        c1_up = F.interpolate(c1, size=(target_h, target_w),
                              mode='bilinear', align_corners=False)
        c1_r  = self.c1_reduce(c1_up)
        combined = self.final_conv(torch.cat([FI, c1_r], dim=1))

        # Upsample final a resolución original
        out = F.interpolate(self.head(combined), size=out_size,
                            mode='bilinear', align_corners=False)
        return out

In [ ]:
class TerSeg(nn.Module):
    """
    TerSeg: red dual-branch CNN (ResNet-34) + Transformer (Swin-Tiny).
    
    Canales CNN  (ResNet-34): 64 → 128 → 256 → 512
    Canales Swin (Swin-Tiny): 96 → 192 → 384 → 768
    
    Fusión:
      Stages 1-2 → FL (CNN domina)
      Stages 3-4 → FG (Swin domina)
    
    Decoder FLGA agrega los 4 niveles fusionados.
    """
    def __init__(self, num_classes: int = 4, swin_img_size: int = 224):
        super().__init__()
        self.swin_img_size = swin_img_size

        # --- Branch_C: ResNet-34 ---
        rn = resnet34(weights=ResNet34_Weights.DEFAULT)
        self.c_stem   = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.c_layer1 = rn.layer1   # 64ch,  H/4
        self.c_layer2 = rn.layer2   # 128ch, H/8
        self.c_layer3 = rn.layer3   # 256ch, H/16
        self.c_layer4 = rn.layer4   # 512ch, H/32

        # --- Branch_T: Swin-Tiny (timm) ---
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=True,
            features_only=True,   # devuelve features de los 4 stages
            out_indices=(0, 1, 2, 3)
        )
        # Canales de salida Swin-Tiny: [96, 192, 384, 768]
        swin_chs = [96, 192, 384, 768]
        cnn_chs  = [64, 128, 256, 512]

        # --- Módulos de fusión FL (stages 1-2) y FG (stages 3-4) ---
        # Canales de salida fusionados = canales CNN
        self.fl1 = FLModule(cnn_chs[0], swin_chs[0], cnn_chs[0])
        self.fl2 = FLModule(cnn_chs[1], swin_chs[1], cnn_chs[1])
        self.fg3 = FGModule(cnn_chs[2], swin_chs[2], swin_chs[2])
        self.fg4 = FGModule(cnn_chs[3], swin_chs[3], swin_chs[3])

        # Canales de los 4 niveles fusionados
        fused_chs = [cnn_chs[0], cnn_chs[1], swin_chs[2], swin_chs[3]]

        # --- Decoder FLGA ---
        self.decoder = FLGADecoder(
            in_channels_list = fused_chs,
            c1_ch            = cnn_chs[0],  # feature map de la capa 1 del CNN
            mid_ch           = 128,
            num_classes      = num_classes
        )

    def _swin_forward(self, x):
        """Interpola a 224×224 para Swin pretrained y devuelve los 4 stages.
        Los features se devuelven en formato [B, C, H, W] tras permutar desde [B, H, W, C]."""
        x_swin = F.interpolate(x, size=(self.swin_img_size, self.swin_img_size),
                               mode='bilinear', align_corners=False)
        feats = self.swin(x_swin)  # lista de tensores [B, H', W', C]
        # timm swin devuelve [B, H, W, C] → permutar a [B, C, H, W]
        return [f.permute(0, 3, 1, 2).contiguous() for f in feats]

    def forward(self, x):
        H, W = x.shape[-2:]

        # Branch_C
        c0 = self.c_stem(x)         # H/4
        c1 = self.c_layer1(c0)      # H/4,  64ch
        c2 = self.c_layer2(c1)      # H/8,  128ch
        c3 = self.c_layer3(c2)      # H/16, 256ch
        c4 = self.c_layer4(c3)      # H/32, 512ch

        # Branch_T (con imagen interpolada a 224)
        t1, t2, t3, t4 = self._swin_forward(x)  # [B, C, H', W']

        # Fusión FL (stages 1-2) y FG (stages 3-4)
        f1 = self.fl1(c1, t1)   # cnn_chs[0] = 64ch
        f2 = self.fl2(c2, t2)   # cnn_chs[1] = 128ch
        f3 = self.fg3(c3, t3)   # swin_chs[2] = 384ch
        f4 = self.fg4(c4, t4)   # swin_chs[3] = 768ch

        # Decoder FLGA
        out = self.decoder([f1, f2, f3, f4], c1=c1, out_size=(H, W))
        return out


def build_model():
    return TerSeg(num_classes=NUM_CLASSES, swin_img_size=SWIN_IMG_SIZE).to(DEVICE)


# Verificación
_m = build_model()
_m.eval()
with torch.no_grad():
    _x = torch.randn(2, 3, 256, 256).to(DEVICE)
    _out = _m(_x)
    print(f'Forward OK — salida: {_out.shape}')  # (2, 4, 256, 256)

n_params = count_parameters(_m)
print(f'Parámetros entrenables: {n_params:.2f}M  (referencia KB: ~49.19M)')
del _m, _x, _out

## 4. Loss, Optimizer y Scheduler

In [ ]:
def criterion_fn():
    return FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA,
                     ignore_index=IGNORE_INDEX)


def optimizer_fn(params):
    return torch.optim.Adam(params, lr=LR)


def scheduler_fn(optimizer):
    # ReduceLROnPlateau en val mIoU: reduce lr si no mejora en 5 epochs
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=5, factor=0.5, verbose=True
    )


print('✅ Loss: FocalLoss (α=0.25, γ=2.0)')
print('  Optimizer: Adam (lr=1e-4)')
print('  Scheduler: ReduceLROnPlateau (mode=max, patience=5, factor=0.5)')
print()
print('  ⚠️ Nota: ReduceLROnPlateau requiere que run_multi_seed pase val_miou')
print('     al scheduler. Verificar que mars_utils v4 soporta este modo.')

## 5. Entrenamiento Multi-Seed

In [ ]:
summary = run_multi_seed(
    model_fn       = build_model,
    df_train       = df_train,
    df_val         = df_val,
    df_gold        = df_gold,
    criterion_fn   = criterion_fn,
    optimizer_fn   = optimizer_fn,
    scheduler_fn   = scheduler_fn,
    model_name     = MODEL_NAME,
    device         = DEVICE,
    num_epochs     = MAX_EPOCHS,
    patience       = PATIENCE,
    batch_size     = BATCH_SIZE,
    fast_subset    = FAST_SUBSET,
    n_train_fast   = 200,
    n_val_fast     = 50,
)

## 6. Resultados Agregados

In [ ]:
print_summary_table(summary)

In [ ]:
plot_best_seed_curves(summary)

In [ ]:
params_M = count_parameters(build_model())
append_benchmark_results(summary, params_M=params_M)
print('✅ Resultados guardados en results/benchmark_results.csv')

## 7. Visualización Cualitativa

In [ ]:
best_seed = max(summary["per_seed"], key=lambda r: r["mIoU"])["seed"]
print(f"Mejor seed: {best_seed} | mIoU gold: {max(summary['per_seed'], key=lambda r: r['mIoU'])['mIoU']:.4f}")

best_model = build_model().to(DEVICE)
ckpt = torch.load(
    CHECKPOINTS_DIR / f"{MODEL_NAME}_seed{best_seed}_best.pth",
    map_location=DEVICE,
)
best_model.load_state_dict(ckpt["model_state"])
best_model.eval()

visualize_predictions(best_model, df_gold, DEVICE, mean=mean, std=std, n=5)

## 8. Resumen Final

| Campo | Valor |
|-------|-------|
| Modelo | TerSeg |
| Paper | Fan et al., Expert Systems with Applications 270 (2025) |
| Branch CNN | ResNet-34 pretrained |
| Branch Transformer | Swin-Tiny (timm) |
| Fusión | FL (stages 1-2) + FG (stages 3-4) |
| Decoder | FLGA multi-escala |
| Loss | FocalLoss (α=0.25, γ=2.0) |
| Optimizer | Adam (lr=1e-4) |
| Scheduler | ReduceLROnPlateau (patience=5, factor=0.5) |
| Swin input | Interpolado a 224×224 internamente |
| Referencia histórica (2.1k imgs) | mIoU = 0.7498 ± 0.0089 |

---
*Resultados del gold set exportados a `results/benchmark_results.csv`.*